## 1- Kütüphaneler

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer # BoW yöntemi için gerekli kütüphane
import re

## 2- Veri Seti

In [9]:
# Veri setimizi yükleyelim

df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## 3- Metin Ön İşleme - Veri Temizleme

In [10]:
# Metin verileriimizi alalım

comments = df["review"][:100] # 50k veri olduğu için uzun sürecekti. Bu yüzden ilk 100 veri ile çalışıyoruz
labels = df["sentiment"][:100] # 50k veri olduğu için uzun sürecekti. Bu yüzden ilk 100 veri ile çalışıyoruz

In [ ]:
# Metin temizleme fonksiyonu
# Metin temizleme işlemlerini gerçekleştireceğimiz bir fonksiyon yazacacağız

def clean_text(text):
    # küçük harfe çevirme
    text = text.lower()

    # rakamları silelim
    text = re.sub(r"\d+", "", text) # \d - Regex'te bir rakam karakterini temsil eder. Yani 0 ile 9 arasındaki herhangi bir sayı.
                                    # + - - bir veya daha fazla eşleşme anlamına gelir. Yani birden fazla rakam yan yana gelmişse hepsini tek seferde yakalar. (2025 gibi)
    
    # özel karakterleri silelim
    text = re.sub(r"[^\w\s]", "", text) # ^ - Tersleme anlamına gelir. Yani köşeli parantez içindeki karakterler dışındaki karakterleri seçer.
                                       # \w - Kelime karakterini temsil eder. Yani harfler (a-z, A-Z), rakamlar (0-9) ve alt çizgi (_) karakterlerini kapsar.
                                       # \s - Boşluk karakterini temsil eder. Yani boşluk, tab, yeni satır gibi boşluk karakterlerini kapsar.
                                       # Yani bu ifade, kelime karakterleri ve boşluk karakterleri dışındaki tüm karakterleri seçer ve siler.

    # "a, I" gibi tek karakterli kelimeleri silelim
    text = " ".join([word for word in text.split() if len(word) > 2])

    # stop words'leri silelim (sonradan güncellediğimiz kısım)
    import nltk
    from nltk.corpus import stopwords
    nltk.download("stopwords")
    stop_words = set(stopwords.words("english"))
    text = " ".join([word for word in text.split() if word not in stop_words])  # <-- liste yerine string

    return text

In [18]:
#  text = " ".join([word for word in text.split() if len(word) > 2]) fonksiyonu nasıl çalışır?

# deneme_text = "merhaba t dünya"
# print(deneme_text.split())

# [word for word in deneme_text.split() if len(word) > 2] # tek karaterli kelime olan "t"yi almaz

In [19]:
# Metin temizleme işlemini gerçekleştirelim fonksiyonumuzu kullanarak

cleaned_comments = [clean_text(comment) for comment in comments]

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\BedirhanOrseloglu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\BedirhanOrseloglu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\BedirhanOrseloglu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\BedirhanOrseloglu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\BedirhanOrseloglu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\BedirhanOrseloglu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\

In [20]:
# Hem eski hem de temizlenmiş metinlere bakalım

for i in range(5):
    print(f"Eski Metin: {comments[i]}")
    print(f"Temizlenmiş Metin: {cleaned_comments[i]}")
    print()

Eski Metin: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is du

# 4- Feature Engeering - Metin Temisili

In [21]:
# CountVectorizer kütüphanemizi kullanarak Bag of Words (BoW) yöntemimizi ele alalım

# İlk olarak nesnemii oluşturalım
vectorizer = CountVectorizer()

# Vektörleştirme işlemini gerçekleştirelim

vectorized_comments = vectorizer.fit_transform(cleaned_comments)

In [22]:
# Kelime kümemizi bulalım
import numpy as np, sys
np.set_printoptions(threshold=sys.maxsize, linewidth=sys.maxsize)  # çıktıda vektörün tamamını görebilmek için ekledik

feature_names = vectorizer.get_feature_names_out()  # Kelime kümesindeki kelimeleri görelim
feature_names

array(['abbot', 'abbreviated', 'abetted', 'abiding', 'ability', 'able', 'aboveaverage', 'abraham', 'abrahams', 'absolute', 'absolutely', 'absorb', 'abstracted', 'absurd', 'abusive', 'academy', 'accent', 'accented', 'accents', 'accentuate', 'accepted', 'accepting', 'accepts', 'accident', 'accidentally', 'accomplish', 'accomplished', 'accomplishes', 'according', 'account', 'accountant', 'accounts', 'accurate', 'accuses', 'accustomed', 'achieve', 'achieved', 'achievement', 'achingly', 'acolyte', 'acquired', 'across', 'acroyd', 'act', 'acting', 'action', 'actions', 'activate', 'actor', 'actors', 'actress', 'actresses', 'acts', 'actual', 'actually', 'ada', 'adabr', 'adams', 'adapted', 'adas', 'add', 'added', 'addict', 'addiction', 'adding', 'addition', 'address', 'adequate', 'adjusterbr', 'admitted', 'adnausem', 'adrian', 'adult', 'advent', 'adventure', 'adventureoh', 'advise', 'affair', 'affairbr', 'affected', 'affection', 'affects', 'afraid', 'africa', 'aftermath', 'afternoons', 'afterwar

In [23]:
len(feature_names)  # kelime kümemizde 4889 adet benzersiz kelimemiz varmış.

4796

In [24]:
# Vektör temsilimizi oluşturalım
import numpy as np, sys
np.set_printoptions(threshold=sys.maxsize, linewidth=sys.maxsize)  # çıktıda vektörün tamamını görebilmek için ekledik

print(f"Vektör temsili: {vectorized_comments.toarray()[:2]}") # ilk 2 cümlemize baktık

# Bu çıktıda 2 satır (yorum) 4889 sütun (kelime kümemiz) var
# Örneğin ilk cümlede 6 numaralı kelimemiz (about) varmış (kelimeyi bulmak için kelime kümemizi inceleyelebiliriz.)

Vektör temsili: [[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 

In [25]:
# Daha güzel inceleyebilmek için DataFrame'e çevirelim

df_bow = pd.DataFrame(vectorized_comments.toarray(), columns=feature_names)
df_bow.head()

# ilk cümlede "about" kelimesinin olduğunu yine görebiliyoruz (0. satır ve about sütunu kesişimi)


,abbot,abbreviated,abetted,abiding,ability,able,aboveaverage,abraham,abrahams,absolute,...,zany,zellweger,zerog,zeus,zombie,zombiebr,zone,zoo,zooms,zwick
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,1,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# # Yukarıdaki df'in tüm sütunları görmek istersek

# df_bow = pd.DataFrame(vectorized_comments.toarray(), columns=feature_names)

# # Tüm sütunların kesilmeden gösterilmesi için pandas ayarları
# import pandas as pd
# from IPython.display import display
# pd.set_option('display.max_columns', None)
# pd.set_option('display.width', None)
# pd.set_option('display.max_colwidth', None)

# # İlk 5 satırı tüm sütunlarla gör
# display(df_bow.head())


In [27]:
# Her bir kelimenin frekansını bulalım
word_counts = vectorized_comments.sum(axis=0).A1  # A1 ile 2D array'ı 1D array'a çeviriyoruz
word_counts

array([  1,   1,   2,   2,   1,   2,   1,   1,   1,   1,   7,   1,   1,   2,   1,   1,   2,   1,   1,   1,   3,   1,   1,   2,   1,   1,   1,   1,   2,   1,   1,   1,   3,   1,   1,   1,   1,   1,   1,   1,   1,   2,   1,   7,  27,   6,   1,   1,   9,  19,   6,   2,   1,   4,  16,   4,   1,   1,   1,   2,   1,   2,   1,   1,   3,   2,   1,   1,   1,   1,   1,   2,   2,   1,   5,   1,   1,   3,   1,   3,   1,   1,   2,   1,   4,   1,   1,   1,   1,   1,   2,   1,   1,   2,   2,   1,   1,   2,   1,   2,   1,   1,   1,   1,   1,   1,   1,   1,   1,   4,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   2,   1,   3,   1,   1,   1,   1,   1,   3,   1,   1,   1,   1,   1,   2,   1,   1,   1,   1,   9,   2,   7,   1,   3,  30,  11,   3,   2,   5,   2,   4,   1,   1,   1,   3,   4,   1,   1,   1,   5,   2,   3,   2,   4,   2,   1,   1,   1,   2,   1,   1,   1,   1,   2,   1,   1,   1,   1,   1,   1,   1,   2,   2,   1,   4,  19,   1,   1,   1,   1,   1,   1,   1,   1,   2,   1,   1,   2,   2

In [ ]:
# Yukarıda hangi kelimenin kaç kere geçtiğini karmaşık bir şekilde görüyoruz. Bunu daha güzel bir şekilde görelim

word_freq = dict(zip(feature_names , word_counts))
word_freq

# Bunları frekansına göre sıralayalım
sorted_word_freq = dict(sorted(word_freq.items(), key=lambda item: item[1], reverse=True))  # item[1] değere (frekans) göre sıralama yapar, item[0] ise anahtara (kelimeler) göre sıralama yapar
sorted_word_freq



# !!!!!!!!!!!!!!

# STOP WORDS'LERİN KALDIRIMI:

# Gördüğümüz gibi the: 1378, and: 645, this: 258 gibi kelimeler en çok geçen kelimeler olmuş ki biz şu an sadaece ilk 100 yorum ile çalışıyoruz.
# Bu kelimeleri stop-word olarak adlandırıyorduk ve genellikle metin madenciliği ve doğal dil işleme görevlerinde çıkarılırlar.
# Çünkü bu tür kelimeler genellikle anlam taşımazlar ve modelin performansını olumsuz etkileyebilirler.

# !!!!!
# Bunun için "clean_text" fonksiyonumuzu GÜNCELLEDİK buraya kadar tekar çalıştırdık

{'movie': 169,
 'film': 127,
 'one': 100,
 'like': 80,
 'even': 58,
 'good': 54,
 'see': 52,
 'story': 45,
 'would': 43,
 'movies': 40,
 'much': 40,
 'really': 39,
 'bad': 36,
 'first': 36,
 'time': 35,
 'could': 34,
 'get': 33,
 'life': 33,
 'way': 33,
 'dont': 32,
 'never': 31,
 'also': 30,
 'seen': 30,
 'scene': 29,
 'well': 29,
 'man': 28,
 'young': 28,
 'acting': 27,
 'made': 27,
 'many': 27,
 'character': 26,
 'people': 26,
 'scenes': 26,
 'still': 26,
 'end': 25,
 'films': 25,
 'make': 25,
 'best': 24,
 'characters': 24,
 'plot': 24,
 'find': 23,
 'going': 23,
 'little': 23,
 'least': 22,
 'pretty': 22,
 'say': 22,
 'think': 22,
 'two': 22,
 'war': 22,
 'cant': 21,
 'lot': 21,
 'love': 21,
 'nothing': 21,
 'old': 21,
 'something': 21,
 'better': 20,
 'however': 20,
 'ive': 20,
 'three': 20,
 'watch': 20,
 'worst': 20,
 'actors': 19,
 'another': 19,
 'back': 19,
 'know': 19,
 'new': 19,
 'thing': 19,
 'gets': 18,
 'got': 18,
 'great': 18,
 'mario': 18,
 'though': 18,
 'years': 18